# LangChain Expression Language (LCEL)

LangChain Expression Language (LCEL) is a declarative way to compose and chain modular AI components like prompts, models, and parsers using the pipe operator (|).

### Flow of llm
```
Input -> Acrion -> LLMs -> Output
```
- LCEL takes place in Action component.

### How LCEL Works
- Pipe Syntax: Connects components sequentially where the output of one step feeds directly into the next (e.g., prompt | model | parser).

- Runnable Protocol: Built on a standard interface supporting .invoke(), .stream(), and .batch() for unified execution.

- Built-in Features: Automatically handles asynchronous operations, parallel execution, streaming, and retries out of the box.

- pass output of one command to input of other component
```py
chain = prompt | llm | output_parser
```

### Basic Template

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.
Answer the following question: {question}
"""
)

llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="deepseek/deepseek-v4-flash"
)
output_parser = StrOutputParser()

### Simple Chain

In [2]:
chain = prompt | llm | output_parser

print(chain.input_schema.schema())
print(chain.output_schema.schema())

# Synchronous invocation (waits for the full response)
result = chain.invoke({"question": "Tell me about The Godfather Movie"})
print(result)

/tmp/ipykernel_21872/1077562779.py:3: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(chain.input_schema.schema())


{'properties': {'question': {'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'PromptInput', 'type': 'object'}
{'title': 'StrOutputParserOutput', 'type': 'string'}
**The Godfather** (1972) is an American crime drama directed by **Francis Ford Coppola** and based on **Mario Puzo’s** 1969 novel. It is widely considered one of the greatest films ever made and is the first installment in *The Godfather* trilogy.

**Main cast:**
- **Marlon Brando** as Vito Corleone, the aging patriarch of a powerful Mafia family
- **Al Pacino** as Michael Corleone, Vito’s reluctant youngest son who becomes the family’s new leader
- **James Caan** as Sonny Corleone
- **Robert Duvall** as Tom Hagen
- **Diane Keaton** as Kay Adams
- **Talia Shire** as Connie Corleone
- **John Cazale** as Fredo Corleone

**Plot overview:**  
The film follows the Corleone family, one of New York’s most powerful organized-crime dynasties. When Vito Corleone refuses to enter the narcotics trade, he is ta

### Streaming

In [3]:
# Iterate over chunks as they arrive from the LLM
for chunk in chain.stream({"question": "Tell me about The Godfather Movie"}):
    print(chunk)

















































































































































































































































































































































































































































































































































































































































































































































































































































































































































































**
The
 God
father
 (
197
2
)**
 is
 an
 American
 crime

### Parallel batch streaming

In [4]:
# Prepare multiple inputs for batch processing
inputs = [
    {"question": "Tell me about The Godfather Movie"},
    {"question": "Give a short summary of 1984 by George Orwell"}
]

# Execute the chain in batch and print each result
results = chain.batch(inputs)
for r in results:
    print(r)

**The Godfather** (1972) is an American crime drama directed by **Francis Ford Coppola** and based on **Mario Puzo’s** 1969 novel. It is widely considered one of the greatest films ever made and is the first installment in *The Godfather* trilogy.

**Main cast:**
- **Marlon Brando** as Vito Corleone, the aging patriarch of a powerful Mafia family
- **Al Pacino** as Michael Corleone, Vito’s reluctant youngest son who becomes the family’s new leader
- **James Caan** as Sonny Corleone
- **Robert Duvall** as Tom Hagen
- **Diane Keaton** as Kay Adams
- **Talia Shire** as Connie Corleone
- **John Cazale** as Fredo Corleone

**Plot overview:**  
The film follows the Corleone family, one of New York’s most powerful organized-crime dynasties. When Vito Corleone refuses to enter the narcotics trade, he is targeted by a rival faction. After an assassination attempt leaves him wounded, his son Michael—initially seen as an outsider—steps into the family business. Michael’s transformation from a war

# Build-In Runnables in LCEL

Runnables in LangChain are standardized units of work that take an input, process it, and return an output using a shared interface.

1. RunnablePassThrough
2. RunnableLambda
3. RunableParallel

### RunnablePassthrough

- RunnablePassthrough is the simplest runnable that just returns the input as-is, without doing anything to it.

UseCase:
- Pass user input as it is to chain.

### Runnable Lambda

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.
Answer the following question: {question}
"""
)

llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="gpt-oss-120b"
)
output_parser = StrOutputParser()

def get_len(text):
    return len(text)

chain = prompt | llm | output_parser | RunnableLambda(get_len)

print(chain.input_schema.schema())
print(chain.output_schema.schema())

### RunnableParallel

In [7]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.
Answer the following question: {question}
"""
)

llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="gpt-oss-120b"
)
output_parser = StrOutputParser()

chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "operation_b": prompt | llm | output_parser
    }
)

print(chain.invoke({"question": "What is the capital of France?"}))
print(chain.input_schema.schema())
print(chain.output_schema.schema())


{'operation_a': {'question': 'What is the capital of France?'}, 'operation_b': 'The capital of France is **Paris**.'}
{'properties': {'root': {'title': 'Root'}, 'question': {'title': 'Question', 'type': 'string'}}, 'required': ['root', 'question'], 'title': 'RunnableParallel<operation_a,operation_b>Input', 'type': 'object'}
{'properties': {'operation_a': {'title': 'Operation A'}, 'operation_b': {'title': 'Operation B', 'type': 'string'}}, 'required': ['operation_a', 'operation_b'], 'title': 'RunnableParallel<operation_a,operation_b>Output', 'type': 'object'}


/tmp/ipykernel_21872/2467563903.py:25: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(chain.input_schema.schema())
/tmp/ipykernel_21872/2467563903.py:26: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(chain.output_schema.schema())


### Runnable Passthrough

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

model = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="gpt-oss-120b"
)

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("What is dswithbappy?")

/home/ajayp/anaconda3/envs/langchain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1467.78it/s]


'dswithbappy is a content provider that focuses on delivering material about Data Science, artificial intelligence, machine learning, deep learning, computer vision, natural language processing, Python programming, and related topics, all in English.'

# More on LCEL

1. Combine chain

```py
composed_chain = {"politician": chain} | historian_prompt | model | StrOutputParser()
```